In [ ]:
# Check database structure

In [1]:
import sqlite3
import pandas as pd

DB_PATH = r"D:\AI_IT_HELPDESK\database\helpdesk.db"

conn = sqlite3.connect(DB_PATH)

for table in ["tickets", "users", "audit_logs"]:

    print("\n" + "=" * 50)
    print("TABLE:", table)
    print("=" * 50)

    structure = pd.read_sql_query(
        f"PRAGMA table_info({table})",
        conn
    )

    display(structure)

conn.close()


TABLE: tickets


,cid,name,type,notnull,dflt_value,pk
0,0,ticket_id,INTEGER,0,NaN,1
1,1,user_id,INTEGER,0,NaN,0
2,2,title,TEXT,1,NaN,0
3,3,description,TEXT,0,NaN,0
4,4,category_id,INTEGER,0,NaN,0
5,5,priority,TEXT,0,'Medium',0
6,6,status,TEXT,0,'Open',0
7,7,created_at,DATETIME,0,CURRENT_TIMESTAMP,0
8,8,updated_at,DATETIME,0,CURRENT_TIMESTAMP,0



TABLE: users


,cid,name,type,notnull,dflt_value,pk
0,0,user_id,INTEGER,0,NaN,1
1,1,username,TEXT,1,NaN,0
2,2,email,TEXT,0,NaN,0
3,3,password_hash,TEXT,0,NaN,0
4,4,role,TEXT,0,'user',0
5,5,created_at,DATETIME,0,CURRENT_TIMESTAMP,0



TABLE: audit_logs


,cid,name,type,notnull,dflt_value,pk
0,0,log_id,INTEGER,0,NaN,1
1,1,user_id,INTEGER,0,NaN,0
2,2,ticket_id,INTEGER,0,NaN,0
3,3,action,TEXT,1,NaN,0
4,4,details,TEXT,0,NaN,0
5,5,created_at,DATETIME,0,CURRENT_TIMESTAMP,0


In [ ]:
#Check ticket status distribution

In [2]:
conn = sqlite3.connect(DB_PATH)

status_df = pd.read_sql_query(
    """
    SELECT status, COUNT(*) AS total
    FROM tickets
    GROUP BY status
    """,
    conn
)

display(status_df)

conn.close()


,status,total
0,Open,4


In [3]:
import sqlite3
import pandas as pd

DB_PATH = r"D:\AI_IT_HELPDESK\database\helpdesk.db"

conn = sqlite3.connect(DB_PATH)

for table in ["tickets", "users", "audit_logs"]:
    print("\nTABLE:", table)
    display(pd.read_sql_query(
        f"PRAGMA table_info({table})", conn
    ))

conn.close()



TABLE: tickets


,cid,name,type,notnull,dflt_value,pk
0,0,ticket_id,INTEGER,0,NaN,1
1,1,user_id,INTEGER,0,NaN,0
2,2,title,TEXT,1,NaN,0
3,3,description,TEXT,0,NaN,0
4,4,category_id,INTEGER,0,NaN,0
5,5,priority,TEXT,0,'Medium',0
6,6,status,TEXT,0,'Open',0
7,7,created_at,DATETIME,0,CURRENT_TIMESTAMP,0
8,8,updated_at,DATETIME,0,CURRENT_TIMESTAMP,0



TABLE: users


,cid,name,type,notnull,dflt_value,pk
0,0,user_id,INTEGER,0,NaN,1
1,1,username,TEXT,1,NaN,0
2,2,email,TEXT,0,NaN,0
3,3,password_hash,TEXT,0,NaN,0
4,4,role,TEXT,0,'user',0
5,5,created_at,DATETIME,0,CURRENT_TIMESTAMP,0



TABLE: audit_logs


,cid,name,type,notnull,dflt_value,pk
0,0,log_id,INTEGER,0,NaN,1
1,1,user_id,INTEGER,0,NaN,0
2,2,ticket_id,INTEGER,0,NaN,0
3,3,action,TEXT,1,NaN,0
4,4,details,TEXT,0,NaN,0
5,5,created_at,DATETIME,0,CURRENT_TIMESTAMP,0


In [4]:
conn = sqlite3.connect(DB_PATH)

status_df = pd.read_sql_query(
    """
    SELECT status, COUNT(*) AS total
    FROM tickets
    GROUP BY status
    """,
    conn
)

display(status_df)

conn.close()


,status,total
0,Open,4


In [5]:
import sqlite3
import pandas as pd

DB_PATH = r"D:\AI_IT_HELPDESK\database\helpdesk.db"

with sqlite3.connect(DB_PATH) as conn:

    for table in ["tickets", "users", "audit_logs"]:
        print(f"\n--- {table.upper()} TABLE ---")
        display(pd.read_sql_query(
            f"PRAGMA table_info({table})", conn
        ))

    print("\n--- TICKET STATUS SUMMARY ---")

    display(pd.read_sql_query(
        """
        SELECT status, COUNT(*) AS total
        FROM tickets
        GROUP BY status
        """,
        conn
    ))



--- TICKETS TABLE ---


,cid,name,type,notnull,dflt_value,pk
0,0,ticket_id,INTEGER,0,NaN,1
1,1,user_id,INTEGER,0,NaN,0
2,2,title,TEXT,1,NaN,0
3,3,description,TEXT,0,NaN,0
4,4,category_id,INTEGER,0,NaN,0
5,5,priority,TEXT,0,'Medium',0
6,6,status,TEXT,0,'Open',0
7,7,created_at,DATETIME,0,CURRENT_TIMESTAMP,0
8,8,updated_at,DATETIME,0,CURRENT_TIMESTAMP,0



--- USERS TABLE ---


,cid,name,type,notnull,dflt_value,pk
0,0,user_id,INTEGER,0,NaN,1
1,1,username,TEXT,1,NaN,0
2,2,email,TEXT,0,NaN,0
3,3,password_hash,TEXT,0,NaN,0
4,4,role,TEXT,0,'user',0
5,5,created_at,DATETIME,0,CURRENT_TIMESTAMP,0



--- AUDIT_LOGS TABLE ---


,cid,name,type,notnull,dflt_value,pk
0,0,log_id,INTEGER,0,NaN,1
1,1,user_id,INTEGER,0,NaN,0
2,2,ticket_id,INTEGER,0,NaN,0
3,3,action,TEXT,1,NaN,0
4,4,details,TEXT,0,NaN,0
5,5,created_at,DATETIME,0,CURRENT_TIMESTAMP,0



--- TICKET STATUS SUMMARY ---


,status,total
0,Open,4


In [ ]:
# Add Ticket Assignment Column

In [6]:
import sqlite3

DB_PATH = r"D:\AI_IT_HELPDESK\database\helpdesk.db"

conn = sqlite3.connect(DB_PATH)

try:
    cursor = conn.cursor()

    columns = [
        row[1]
        for row in cursor.execute(
            "PRAGMA table_info(tickets)"
        ).fetchall()
    ]

    if "assigned_to" not in columns:

        cursor.execute(
            """
            ALTER TABLE tickets
            ADD COLUMN assigned_to INTEGER
            REFERENCES users(user_id)
            """
        )

        conn.commit()
        print("assigned_to column added successfully!")

    else:
        print("assigned_to column already exists.")

except Exception:
    conn.rollback()
    raise

finally:
    conn.close()


assigned_to column added successfully!


In [ ]:
# Verify Assignment Column

In [7]:
conn = sqlite3.connect(DB_PATH)

columns_df = pd.read_sql_query(
    "PRAGMA table_info(tickets)",
    conn
)

display(columns_df)

conn.close()


,cid,name,type,notnull,dflt_value,pk
0,0,ticket_id,INTEGER,0,NaN,1
1,1,user_id,INTEGER,0,NaN,0
2,2,title,TEXT,1,NaN,0
3,3,description,TEXT,0,NaN,0
4,4,category_id,INTEGER,0,NaN,0
5,5,priority,TEXT,0,'Medium',0
6,6,status,TEXT,0,'Open',0
7,7,created_at,DATETIME,0,CURRENT_TIMESTAMP,0
8,8,updated_at,DATETIME,0,CURRENT_TIMESTAMP,0
9,9,assigned_to,INTEGER,0,NaN,0


In [ ]:
# Prepare Admin Dashboard Data

In [8]:
conn = sqlite3.connect(DB_PATH)

dashboard_df = pd.read_sql_query(
    """
    SELECT
        t.ticket_id,
        t.title,
        t.description,
        u.username AS created_by,
        a.username AS assigned_to,
        c.category_name,
        t.priority,
        t.status,
        t.created_at,
        t.updated_at

    FROM tickets t

    LEFT JOIN users u
        ON t.user_id = u.user_id

    LEFT JOIN users a
        ON t.assigned_to = a.user_id

    LEFT JOIN categories c
        ON t.category_id = c.category_id

    ORDER BY t.ticket_id DESC
    """,
    conn
)

display(dashboard_df)

conn.close()


,ticket_id,title,description,created_by,assigned_to,category_name,priority,status,created_at,updated_at
0,4,scanner not working,scanner not working,admin,None,Software,Medium,Open,2026-11-02 10:15:57,2026-11-02 10:15:57
1,3,my account not working,my account not working,user,None,Account & Password,High,Open,2026-11-02 10:14:56,2026-11-02 10:14:56
2,2,printer not working,printer not working,NaN,None,Printer,Low,Open,2026-11-02 09:22:50,2026-11-02 09:22:50
3,1,Printer is offline,The network printer is showing offline status.,NaN,None,Printer,Medium,Open,2026-11-02 09:12:43,2026-11-02 09:12:43


In [10]:

from pathlib import Path
import os
import sys
import subprocess

BASE_DIR = Path(r"D:\AI_IT_HELPDESK")
APP_PATH = BASE_DIR / "frontend" / "app.py"

print("Application path:", APP_PATH)
print("File exists:", APP_PATH.exists())

if not APP_PATH.exists():
    raise FileNotFoundError(
        f"Streamlit application not found: {APP_PATH}"
    )


Application path: D:\AI_IT_HELPDESK\frontend\app.py
File exists: True


In [11]:
process = subprocess.Popen(
    [
        sys.executable,
        "-m",
        "streamlit",
        "run",
        str(APP_PATH),
        "--server.port=8501"
    ],
    cwd=str(APP_PATH.parent)
)

print("Streamlit started!")
print("Process ID:", process.pid)
print("URL: http://localhost:8501")


Streamlit started!
Process ID: 32800
URL: http://localhost:8501


In [ ]:
#Create the ticket comments table

In [12]:
import sqlite3
from pathlib import Path

DB_PATH = Path(r"D:\AI_IT_HELPDESK\database\helpdesk.db")

if not DB_PATH.exists():
    raise FileNotFoundError(f"Database not found: {DB_PATH}")

conn = sqlite3.connect(DB_PATH)

try:
    conn.execute("""
        CREATE TABLE IF NOT EXISTS ticket_comments (
            comment_id INTEGER PRIMARY KEY AUTOINCREMENT,
            ticket_id INTEGER NOT NULL,
            user_id INTEGER NOT NULL,
            comment TEXT NOT NULL,
            created_at DATETIME DEFAULT CURRENT_TIMESTAMP,

            FOREIGN KEY (ticket_id)
                REFERENCES tickets(ticket_id),

            FOREIGN KEY (user_id)
                REFERENCES users(user_id)
        )
    """)

    conn.execute("""
        CREATE INDEX IF NOT EXISTS
        idx_ticket_comments_ticket
        ON ticket_comments(ticket_id)
    """)

    conn.commit()

    print("Ticket comments table created successfully!")

    columns = conn.execute(
        "PRAGMA table_info(ticket_comments)"
    ).fetchall()

    print("\nTable structure:")
    for column in columns:
        print(column[1], "-", column[2])

except Exception as e:
    conn.rollback()
    raise e

finally:
    conn.close()


Ticket comments table created successfully!

Table structure:
comment_id - INTEGER
ticket_id - INTEGER
user_id - INTEGER
comment - TEXT
created_at - DATETIME


In [ ]:
# Test saving and retrieving a comment

In [13]:
import sqlite3
from pathlib import Path

DB_PATH = Path(r"D:\AI_IT_HELPDESK\database\helpdesk.db")

conn = sqlite3.connect(DB_PATH)

try:
    # Verify ticket and user exist
    ticket = conn.execute(
        "SELECT ticket_id FROM tickets WHERE ticket_id = ?",
        (4,)
    ).fetchone()

    user = conn.execute(
        "SELECT user_id, username FROM users WHERE user_id = ?",
        (1,)
    ).fetchone()

    if ticket is None:
        raise ValueError("Ticket ID 4 was not found.")

    if user is None:
        raise ValueError("Admin user ID 1 was not found.")

    # Insert a test comment
    conn.execute("""
        INSERT INTO ticket_comments
        (ticket_id, user_id, comment)
        VALUES (?, ?, ?)
    """, (
        4,
        1,
        "Test message: IT support is reviewing this issue."
    ))

    conn.commit()

    # Retrieve comments
    comments = conn.execute("""
        SELECT
            tc.comment_id,
            tc.ticket_id,
            u.username,
            u.role,
            tc.comment,
            tc.created_at
        FROM ticket_comments tc
        JOIN users u
            ON tc.user_id = u.user_id
        WHERE tc.ticket_id = ?
        ORDER BY tc.comment_id ASC
    """, (4,)).fetchall()

    print("Ticket communication test successful!\n")

    for row in comments:
        print(row)

finally:
    conn.close()


Ticket communication test successful!

(1, 4, 'admin', 'admin', 'Test message: IT support is reviewing this issue.', '2026-11-02 11:02:02')


In [ ]:
# Integrate comments into Streamlit

In [14]:
from pathlib import Path

APP_PATH = Path(r"D:\AI_IT_HELPDESK\frontend\app.py")

print("Application exists:", APP_PATH.exists())
print("Application size:", APP_PATH.stat().st_size, "bytes")

with open(APP_PATH, "r", encoding="utf-8") as file:
    app_code = file.readlines()

print("\nTotal lines:", len(app_code))

for i, line in enumerate(app_code, start=1):
    if any(keyword in line.lower() for keyword in [
        "my tickets",
        "admin dashboard",
        "ticket management",
        "session_state",
        "ticket_id"
    ]):
        print(f"Line {i}: {line.rstrip()}")


Application exists: True
Application size: 20950 bytes

Total lines: 878
Line 67:     return "user" in st.session_state
Line 106:                     st.session_state["user"] = user
Line 107:                     st.session_state["analysis"] = None
Line 123: current_user = st.session_state["user"]
Line 138:         st.session_state.pop(key, None)
Line 151:         "My Tickets",
Line 152:         "Admin Dashboard"
Line 159:         "My Tickets"
Line 283:         ticket_id = cursor.lastrowid
Line 288:             (user_id, ticket_id, action, details)
Line 293:                 ticket_id,
Line 301:         return ticket_id
Line 343:             st.session_state["analysis"] = {
Line 351:     analysis = st.session_state.get("analysis")
Line 436:                     ticket_id = create_support_ticket(
Line 445:                         f"Ticket created successfully! ID: {ticket_id}"
Line 450: # MY TICKETS
Line 453: elif menu == "My Tickets":
Line 462:             SELECT t.ticket_id, t.title, c.c